In [50]:
import pandas as pd
from pathlib import Path

# Define o caminho raiz e o caminho relativo até o dado bruto
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CAMINHO_RAW = BASE_DIR / "data" / "raw" / "industrial_motor_sensor_data_8000.csv"

# ingestão do dataset
df_raw = pd.read_csv(CAMINHO_RAW)

# 1. Dimensão dos dados (linhas, Colunas)
print(f"Dimensões do lote: {df_raw.shape[0]} linhas e {df_raw.shape[1]} colunas\n")

# 2. Tipagem indeferida e consumo de memória
df_raw.info()



Dimensões do lote: 8000 linhas e 5 colunas

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Voltage (V)       8000 non-null   float64
 1   Current (A)       8000 non-null   float64
 2   Temperature (°C)  8000 non-null   float64
 3   Vibration (mm/s)  8000 non-null   float64
 4   Label             8000 non-null   str    
dtypes: float64(4), str(1)
memory usage: 312.6 KB


In [51]:
display(df_raw)

,Voltage (V),Current (A),Temperature (°C),Vibration (mm/s),Label
0,460.01,1.79,118.33,22.37,high
1,419.12,15.52,36.93,2.18,normal
2,380.53,30.78,83.08,12.06,moderate
3,382.82,12.50,54.33,4.48,normal
4,333.79,0.96,118.99,24.41,high
...,...,...,...,...,...
7995,417.74,19.53,86.02,19.48,moderate
7996,393.46,8.80,60.51,1.10,low
7997,366.10,15.47,44.18,0.27,low
7998,411.75,5.09,30.03,7.85,low


In [52]:
df_raw.describe().T

,count,mean,std,min,25%,50%,75%,max
Voltage (V),8000.0,382.502995,56.219001,200.26,356.4700,390.545,410.4800,479.97
Current (A),8000.0,18.368696,12.558719,0.00,10.6100,15.600,26.9625,49.99
Temperature (°C),8000.0,67.222605,26.281466,30.01,44.7575,60.245,89.9875,119.98
Vibration (mm/s),8000.0,10.586649,9.655103,0.00,2.5300,5.070,19.9950,29.99


In [53]:
print(df_raw['Label'].value_counts())
print("\nPercentual:")
print(df_raw['Label'].value_counts(normalize=True) * 100)

Label
high        2000
normal      2000
moderate    2000
low         2000
Name: count, dtype: int64

Percentual:
Label
high        25.0
normal      25.0
moderate    25.0
low         25.0
Name: proportion, dtype: float64


In [54]:
# Métricas centrais (média e desvio padrão) agrupadas pelo Label
metricas_por_classe = df_raw.groupby('Label').agg({
    "Voltage (V)": ["mean", 'std'],
    "Current (A)": ["mean", 'std'],
    "Temperature (°C)": ["mean", "std"],
    "Vibration (mm/s)": ["mean", "std"]
    }).round(2)

print(metricas_por_classe)

         Voltage (V)         Current (A)        Temperature (°C)         \
                mean     std        mean    std             mean    std   
Label                                                                     
high          371.91  104.41       22.27  21.05           104.75   8.65   
low           389.04   17.37       12.09   4.40            52.12  12.09   
moderate      368.90   25.71       24.18   7.76            67.18  18.12   
normal        400.16   11.47       14.93   2.83            44.84   8.74   

         Vibration (mm/s)        
                     mean   std  
Label                            
high                25.05  2.84  
low                  4.35  2.85  
moderate            10.42  6.43  
normal               2.52  1.43  


In [55]:
# Seleciona apenas as colunas numéricas para a correlação
colunas_sensores = ['Voltage (V)', 'Current (A)', 'Temperature (°C)', 'Vibration (mm/s)']
correlacao = df_raw[colunas_sensores].corr().round(3)

print(correlacao)

                  Voltage (V)  Current (A)  Temperature (°C)  Vibration (mm/s)
Voltage (V)             1.000       -0.037            -0.123            -0.131
Current (A)            -0.037        1.000             0.225             0.238
Temperature (°C)       -0.123        0.225             1.000             0.775
Vibration (mm/s)       -0.131        0.238             0.775             1.000


Correlação positiva e alta entre Vibração e Temperatura 

In [56]:
# Identifica zona de transição 
resumo_temp_vib = df_raw.groupby('Label')[['Temperature (°C)', 'Vibration (mm/s)']].describe().T.round(2)

display(resumo_temp_vib)



Label                      high      low  moderate   normal
Temperature (°C) count  2000.00  2000.00   2000.00  2000.00
                 mean    104.75    52.12     67.18    44.84
                 std       8.65    12.09     18.12     8.74
                 min      90.01    30.01     30.01    30.01
                 25%      97.43    41.14     50.63    37.11
                 50%     104.55    53.34     73.91    44.64
                 75%     111.83    63.03     82.15    52.34
                 max     119.98    70.00     89.98    59.99
Vibration (mm/s) count  2000.00  2000.00   2000.00  2000.00
                 mean     25.05     4.35     10.42     2.52
                 std       2.84     2.85      6.43     1.43
                 min      20.01     0.00      0.00     0.00
                 25%      22.56     1.94      3.48     1.27
                 50%      25.14     3.93     12.18     2.52
                 75%      27.42     6.75     15.92     3.76
                 max      29.99    10.00     19.99     5.00

In [57]:
# Limiar crítico
df_critico = df_raw[df_raw["Label"] == "high"]

limiares_criticos = df_critico[["Temperature (°C)", "Vibration (mm/s)"]].agg(["min", "max"]).round(2)

display(limiares_criticos)

,Temperature (°C),Vibration (mm/s)
min,90.01,20.01
max,119.98,29.99


In [58]:
# mapear as escalas completas dos estados

metricas_min_max = df_raw.groupby('Label').agg({
    'Voltage (V)': ['min', 'max'],
    'Current (A)': ['min', 'max'],
    'Temperature (°C)': ['min', 'max'],
    'Vibration (mm/s)': ['min', 'max'] 
})

metricas_min_max

Voltage (V)         Current (A)        Temperature (°C)          \
                 min     max         min    max              min     max   
Label                                                                      
high          200.26  479.97        0.00  49.99            90.01  119.98   
low           360.02  419.97        5.02  20.00            30.01   70.00   
moderate      340.02  419.98       10.00  35.00            30.01   89.98   
normal        380.00  419.98       10.00  19.99            30.01   59.99   

         Vibration (mm/s)         
                      min    max  
Label                             
high                20.01  29.99  
low                  0.00  10.00  
moderate             0.00  19.99  
normal               0.00   5.00

In [59]:
# Criando nova coluna de potência estimada aparente (EPA kVA)

df_processed = df_raw.copy()

df_processed['EPA (kVA)'] = ((df_processed['Voltage (V)'] * df_processed['Current (A)'])/1000).round(3)

display(df_processed)

,Voltage (V),Current (A),Temperature (°C),Vibration (mm/s),Label,EPA (kVA)
0,460.01,1.79,118.33,22.37,high,0.823
1,419.12,15.52,36.93,2.18,normal,6.505
2,380.53,30.78,83.08,12.06,moderate,11.713
3,382.82,12.50,54.33,4.48,normal,4.785
4,333.79,0.96,118.99,24.41,high,0.320
...,...,...,...,...,...,...
7995,417.74,19.53,86.02,19.48,moderate,8.158
7996,393.46,8.80,60.51,1.10,low,3.462
7997,366.10,15.47,44.18,0.27,low,5.664
7998,411.75,5.09,30.03,7.85,low,2.096


In [60]:
# Exportando o dataset processado

CAMINHO_PROCESSED = BASE_DIR / "data" / "processed" / "motores_processados.csv"

df_processed.to_csv(CAMINHO_PROCESSED, index=False)

if CAMINHO_PROCESSED.exists():
    print(f"Arquivo salvo com sucesso em: \n{CAMINHO_PROCESSED.resolve()}")
    print(f"Total de registros exportados: {len(df_processed)} linhas")


Arquivo salvo com sucesso em: 
C:\Users\ghost\Desktop\Projetos IA\analise_motores_industriais\data\processed\motores_processados.csv
Total de registros exportados: 8000 linhas
